In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [18]:
%pip install -q sacrebleu sacremoses

import json
import pandas as pd
from tqdm.auto import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 43.0 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import sys

project_dir = Path("/content/drive/MyDrive/Colab Notebooks/nlp_translation")

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from src.data import load_prepared, make_loaders

config, vocab_fr, vocab_en, prepared = load_prepared(project_dir)

# Réduire la mémoire utilisée
config["batch_size"] = 8

loaders = make_loaders(prepared, config)

train_loader = loaders["train"]
validation_loader = loaders["validation"]
test_loader = loaders["test"]

print("Entraînement :", len(prepared["train"]))
print("Validation :", len(prepared["validation"]))
print("Test :", len(prepared["test"]))
print("Batch size :", config["batch_size"])

Entraînement : 30000
Validation : 2000
Test : 2000
Batch size : 8


In [3]:
print("Dossier du projet :", project_dir)
print("Phrases d'entraînement :", len(prepared["train"]))
print("Phrases de validation :", len(prepared["validation"]))
print("Phrases de test :", len(prepared["test"]))
print("Vocabulaire français :", len(vocab_fr))
print("Vocabulaire anglais :", len(vocab_en))
print("Batch size :", config["batch_size"])

batch = next(iter(train_loader))

print("\nDimensions d'un batch :")
for name in ["src", "src_lengths", "decoder_input", "target"]:
    print(name, ":", tuple(batch[name].shape))

Dossier du projet : /content/drive/MyDrive/Colab Notebooks/nlp_translation
Phrases d'entraînement : 30000
Phrases de validation : 2000
Phrases de test : 2000
Vocabulaire français : 10000
Vocabulaire anglais : 10000
Batch size : 8

Dimensions d'un batch :
src : (8, 39)
src_lengths : (8,)
decoder_input : (8, 58)
target : (8, 58)


# **1. Imports et choix du GPU**

In [4]:
import random
import torch
import torch.nn as nn

from torch.nn.utils.rnn import pack_padded_sequence

PAD_ID = config["special_tokens"]["<PAD>"]

random.seed(config["seed"])
torch.manual_seed(config["seed"])

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Appareil utilisé :", device)

Appareil utilisé : cuda


# **2. Construire l’encodeur, le décodeur et le modèle complet**

In [5]:
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence
)


# 1. Encodeur : conserver les représentations de tous les mots
class EncoderLSTM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 num_layers=1):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=PAD_ID
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

    def forward(self, src, src_lengths):
        embedded = self.embedding(src)

        packed = pack_padded_sequence(
            embedded,
            src_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_outputs, (hidden, cell) = self.lstm(packed)

        # [batch, longueur source, hidden_dim]
        encoder_outputs, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=src.size(1)
        )

        return encoder_outputs, hidden, cell


# 2. Attention additive de Bahdanau
class BahdanauAttention(nn.Module):

    def __init__(self, hidden_dim):
        super().__init__()

        self.encoder_projection = nn.Linear(
            hidden_dim, hidden_dim, bias=False
        )
        self.decoder_projection = nn.Linear(
            hidden_dim, hidden_dim, bias=False
        )
        self.score_projection = nn.Linear(
            hidden_dim, 1, bias=False
        )

    def forward(self, query, encoder_outputs, src_mask):
        # query : dernier état caché du décodeur [batch, hidden_dim]
        # src_mask : True pour les positions valides

        energy = torch.tanh(
            self.encoder_projection(encoder_outputs)
            + self.decoder_projection(query).unsqueeze(1)
        )

        # Un score par position française : [batch, longueur source]
        scores = self.score_projection(energy).squeeze(-1)

        # Empêcher l'attention de regarder le padding
        scores = scores.masked_fill(~src_mask, float("-inf"))

        weights = torch.softmax(scores, dim=-1)

        # Somme pondérée des représentations françaises
        context = torch.bmm(
            weights.unsqueeze(1),
            encoder_outputs
        ).squeeze(1)

        return context, weights


# 3. Décodeur : utiliser le contexte pour prédire le prochain token
class DecoderLSTM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 num_layers=1):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=PAD_ID
        )

        self.attention = BahdanauAttention(hidden_dim)

        # Entrée = embedding du token précédent + contexte
        self.lstm = nn.LSTM(
            embedding_dim + hidden_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.output_layer = nn.Linear(
            hidden_dim * 2,
            vocab_size
        )

    def forward(self, token, hidden, cell,
                encoder_outputs, src_mask):

        embedded = self.embedding(token)

        context, weights = self.attention(
            hidden[-1],
            encoder_outputs,
            src_mask
        )

        lstm_input = torch.cat(
            [embedded, context], dim=-1
        ).unsqueeze(1)

        output, (hidden, cell) = self.lstm(
            lstm_input, (hidden, cell)
        )

        logits = self.output_layer(
            torch.cat([output.squeeze(1), context], dim=-1)
        )

        return logits, hidden, cell, weights


# 4. Modèle complet
class Seq2SeqLSTM(nn.Module):

    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, src_lengths, decoder_input,
                teacher_forcing_ratio=0.5,
                return_attention=False):

        encoder_outputs, hidden, cell = self.encoder(
            src, src_lengths
        )

        src_mask = src.ne(PAD_ID)

        token = decoder_input[:, 0]
        steps = decoder_input.size(1)

        predictions = []
        attention_steps = []

        for t in range(steps):
            logits, hidden, cell, weights = self.decoder(
                token,
                hidden,
                cell,
                encoder_outputs,
                src_mask
            )

            predictions.append(logits)

            if return_attention:
                attention_steps.append(weights)

            if t + 1 < steps:
                if random.random() < teacher_forcing_ratio:
                    token = decoder_input[:, t + 1]
                else:
                    token = logits.argmax(dim=-1)

        predictions = torch.stack(predictions, dim=1)

        if return_attention:
            # [batch, longueur cible, longueur source]
            attention = torch.stack(attention_steps, dim=1)
            return predictions, attention

        return predictions


# 5. Créer le modèle avec les mêmes dimensions que l'expérience 2
random.seed(config["seed"])
torch.manual_seed(config["seed"])

encoder = EncoderLSTM(
    len(vocab_fr),
    config["embedding_dim"],
    config["hidden_dim"],
    config["num_layers"]
)

decoder = DecoderLSTM(
    len(vocab_en),
    config["embedding_dim"],
    config["hidden_dim"],
    config["num_layers"]
)

model = Seq2SeqLSTM(encoder, decoder).to(device)

num_parameters = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Modèle : LSTM avec attention de Bahdanau")
print("Appareil :", device)
print("Paramètres entraînables :", num_parameters)

Modèle : LSTM avec attention de Bahdanau
Appareil : cuda
Paramètres entraînables : 8874000


# **Verification du modele**

In [6]:
batch = next(iter(train_loader))

# Petit test : 2 phrases et 5 étapes de traduction
src = batch["src"][:2].to(device)
src_lengths = batch["src_lengths"][:2]
decoder_input = batch["decoder_input"][:2, :5].to(device)
target = batch["target"][:2, :5].to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

model.eval()

with torch.no_grad():
    logits, attention = model(
        src,
        src_lengths,
        decoder_input,
        teacher_forcing_ratio=1.0,
        return_attention=True
    )

    loss = criterion(
        logits.reshape(-1, len(vocab_en)),
        target.reshape(-1)
    )

# Vérifier les dimensions et l'absence de valeurs non finies
assert logits.shape == (
    src.size(0), decoder_input.size(1), len(vocab_en)
)
assert attention.shape == (
    src.size(0), decoder_input.size(1), src.size(1)
)
assert torch.isfinite(logits).all().item()
assert torch.isfinite(attention).all().item()
assert torch.isfinite(loss).item()

# Pour chaque token anglais, les poids doivent sommer à 1
assert torch.allclose(
    attention.sum(dim=-1),
    torch.ones_like(attention.sum(dim=-1)),
    atol=1e-5
)

# Les positions PAD doivent recevoir un poids nul
pad_mask = src.eq(PAD_ID).unsqueeze(1).expand_as(attention)
pad_weights = attention.masked_select(pad_mask)

if pad_weights.numel() > 0:
    assert torch.all(pad_weights == 0).item()

print("Dimensions des sorties :", tuple(logits.shape))
print("Dimensions de l'attention :", tuple(attention.shape))
print("Loss initiale :", round(loss.item(), 4))
print("Somme des poids d'attention : OK")
print("Masquage du padding : OK")
print("Modèle avec attention vérifié.")

Dimensions des sorties : (2, 5, 10000)
Dimensions de l'attention : (2, 5, 58)
Loss initiale : 9.2048
Somme des poids d'attention : OK
Masquage du padding : OK
Modèle avec attention vérifié.


# **Essai TF 100 %**

In [7]:
import time
import json
import random
from pathlib import Path

import torch
import torch.nn as nn
from tqdm.auto import tqdm


# 1. Configuration de la nouvelle expérience
config = dict(config)

config["teacher_forcing_ratio"] = 1.0
config["epochs"] = 5
config["batch_size"] = 8
config["learning_rate"] = 0.001


# 2. Initialiser les générateurs aléatoires
random.seed(config["seed"])
torch.manual_seed(config["seed"])

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config["seed"])

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

PAD_ID = config["special_tokens"]["<PAD>"]


# 3. Créer un NOUVEAU modèle avec attention
# Les classes avec attention doivent avoir été exécutées avant.
encoder = EncoderLSTM(
    vocab_size=len(vocab_fr),
    embedding_dim=config["embedding_dim"],
    hidden_dim=config["hidden_dim"],
    num_layers=config["num_layers"]
)

decoder = DecoderLSTM(
    vocab_size=len(vocab_en),
    embedding_dim=config["embedding_dim"],
    hidden_dim=config["hidden_dim"],
    num_layers=config["num_layers"]
)

model = Seq2SeqLSTM(encoder, decoder).to(device)

assert hasattr(model.decoder, "attention"), (
    "Exécute d'abord la cellule définissant le modèle avec attention."
)


# 4. Recréer les DataLoaders avec les mêmes données
loaders = make_loaders(prepared, config)

train_loader = loaders["train"]
validation_loader = loaders["validation"]
test_loader = loaders["test"]


# 5. Initialiser un nouvel optimiseur
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config["learning_rate"]
)


# 6. Fonction de perte compatible avec les boucles existantes
# Les tokens PAD ne contribuent pas à la perte.
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
    reduction="sum"
)


# 7. Dossiers dédiés à l'expérience avec teacher forcing à 100 %
checkpoint_dir = (
    project_dir / "checkpoints" / "experience_5_bahdanau_tf100"
)

results_dir = (
    project_dir / "results" / "experience_5_bahdanau_tf100"
)

checkpoint_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)


# 8. Afficher les paramètres de l'expérience
num_parameters = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Modèle : LSTM avec attention de Bahdanau")
print("Appareil :", device)
print("Teacher forcing :", config["teacher_forcing_ratio"])
print("Learning rate :", config["learning_rate"])
print("Batch size :", config["batch_size"])
print("Nombre d'époques :", config["epochs"])
print("Paramètres entraînables :", num_parameters)
print("Sauvegardes :", checkpoint_dir)
print("Résultats :", results_dir)
print("\nNouveau modèle initialisé : prêt pour l'entraînement.")

Modèle : LSTM avec attention de Bahdanau
Appareil : cuda
Teacher forcing : 1.0
Learning rate : 0.001
Batch size : 8
Nombre d'époques : 5
Paramètres entraînables : 8874000
Sauvegardes : /content/drive/MyDrive/Colab Notebooks/nlp_translation/checkpoints/experience_5_bahdanau_tf100
Résultats : /content/drive/MyDrive/Colab Notebooks/nlp_translation/results/experience_5_bahdanau_tf100

Nouveau modèle initialisé : prêt pour l'entraînement.


# **Définir l’entraînement et la validation**

In [8]:
def train_one_epoch(model, loader):

    model.train()

    total_loss = 0.0
    total_tokens = 0

    for batch in tqdm(loader, desc="Entraînement"):

        src = batch["src"].to(device)
        lengths = batch["src_lengths"]  # Rester sur CPU
        decoder_input = batch["decoder_input"].to(device)
        target = batch["target"].to(device)

        optimizer.zero_grad()

        logits = model(
            src,
            lengths,
            decoder_input,
            teacher_forcing_ratio=config["teacher_forcing_ratio"]
        )

        loss_sum = criterion(
            logits.reshape(-1, len(vocab_en)),
            target.reshape(-1)
        )

        valid_tokens = target.ne(PAD_ID).sum().item()
        loss = loss_sum / valid_tokens

        if not torch.isfinite(loss).item():
            raise RuntimeError("Loss non finie : arrêter et vérifier le modèle.")

        loss.backward()

        # Limiter les gradients trop grands
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss_sum.detach().item()
        total_tokens += valid_tokens

    return total_loss / total_tokens


@torch.no_grad()
def validate_one_epoch(model, loader):

    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for batch in tqdm(loader, desc="Validation"):

        src = batch["src"].to(device)
        lengths = batch["src_lengths"]
        decoder_input = batch["decoder_input"].to(device)
        target = batch["target"].to(device)

        logits = model(
            src,
            lengths,
            decoder_input,
            teacher_forcing_ratio=1.0
        )

        loss_sum = criterion(
            logits.reshape(-1, len(vocab_en)),
            target.reshape(-1)
        )

        total_loss += loss_sum.item()
        total_tokens += target.ne(PAD_ID).sum().item()

    return total_loss / total_tokens

# **Lancer l’entraînement et sauvegarder**

In [9]:
history = []
best_validation_loss = float("inf")
total_training_seconds = 0.0

num_parameters = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

for epoch in range(1, config["epochs"] + 1):

    print(f"\nÉpoque {epoch}/{config['epochs']}")

    # Synchroniser le GPU pour mesurer le temps
    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    train_loss = train_one_epoch(model, train_loader)

    if device.type == "cuda":
        torch.cuda.synchronize()

    training_seconds = time.perf_counter() - start
    total_training_seconds += training_seconds

    validation_loss = validate_one_epoch(
        model,
        validation_loader
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": validation_loss,
        "training_seconds": training_seconds,
        "total_training_seconds": total_training_seconds
    }

    history.append(row)

    print(
        f"Train loss : {train_loss:.4f} | "
        f"Validation loss : {validation_loss:.4f} | "
        f"Temps entraînement : {training_seconds / 60:.2f} min"
    )

    improved = validation_loss < best_validation_loss

    if improved:
        best_validation_loss = validation_loss

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": config,
        "vocab_fr": vocab_fr,
        "vocab_en": vocab_en,
        "validation_loss": validation_loss,
        "best_validation_loss": best_validation_loss,
        "num_parameters": num_parameters,
        "history": history,
        "python_rng_state": random.getstate(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_states": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available() else []
        )
    }

    # Sauvegarde de la dernière époque
    torch.save(
        checkpoint,
        checkpoint_dir / "lstm_last.pt"
    )

    # Sauvegarde du meilleur modèle
    if improved:
        torch.save(
            checkpoint,
            checkpoint_dir / "lstm_best.pt"
        )
        print("Meilleur modèle sauvegardé.")

    # Sauvegarde des résultats après chaque époque
    (results_dir / "lstm_history.json").write_text(
        json.dumps(history, indent=2),
        encoding="utf-8"
    )

print("\nEntraînement terminé.")
print("Meilleure loss de validation :", best_validation_loss)
print("Temps total d'entraînement :", total_training_seconds / 60, "min")


Époque 1/5


Entraînement:   0%|          | 0/3750 [00:00<?, ?it/s]

Validation:   0%|          | 0/250 [00:00<?, ?it/s]

Train loss : 5.1730 | Validation loss : 4.6579 | Temps entraînement : 8.92 min
Meilleur modèle sauvegardé.

Époque 2/5


Entraînement:   0%|          | 0/3750 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/250 [00:00<?, ?it/s]

Train loss : 2.9186 | Validation loss : 4.2138 | Temps entraînement : 9.30 min

Époque 5/5


Entraînement:   0%|          | 0/3750 [00:00<?, ?it/s]

Validation:   0%|          | 0/250 [00:00<?, ?it/s]

Train loss : 2.4993 | Validation loss : 4.2829 | Temps entraînement : 9.27 min

Entraînement terminé.
Meilleure loss de validation : 4.190028785101614
Temps total d'entraînement : 45.76818299959999 min


# **Calculer le BLEU et sauvegarder les résultats**

In [11]:
%pip install -q sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 11.0 MB/s eta 0:00:00


In [12]:
import pandas as pd
from sacrebleu.metrics import BLEU

SOS_ID = config["special_tokens"]["<SOS>"]
EOS_ID = config["special_tokens"]["<EOS>"]
PAD_ID = config["special_tokens"]["<PAD>"]

id_to_token_en = {
    index: token for token, index in vocab_en.items()
}

MAX_GENERATION_STEPS = 100


@torch.no_grad()
def generate_validation_batch(eval_model, src, src_lengths):
    eval_model.eval()

    encoder_outputs, hidden, cell = eval_model.encoder(
        src, src_lengths
    )
    src_mask = src.ne(PAD_ID)

    token = torch.full(
        (src.size(0),),
        SOS_ID,
        dtype=torch.long,
        device=src.device
    )

    finished = torch.zeros(
        src.size(0), dtype=torch.bool, device=src.device
    )

    generated = []

    for _ in range(MAX_GENERATION_STEPS):
        logits, hidden, cell, _ = eval_model.decoder(
            token, hidden, cell, encoder_outputs, src_mask
        )

        token = logits.argmax(dim=-1)

        token = torch.where(
            finished,
            torch.full_like(token, EOS_ID),
            token
        )

        generated.append(token.cpu())
        finished |= token.eq(EOS_ID)

        if finished.all().item():
            break

    sequences = torch.stack(generated, dim=1).tolist()
    translations = []

    for sequence in sequences:
        words = []

        for index in sequence:
            if index == EOS_ID:
                break
            if index not in {PAD_ID, SOS_ID}:
                words.append(
                    id_to_token_en.get(index, "<UNK>")
                )

        translations.append(" ".join(words))

    return translations, finished.cpu().tolist()

In [13]:
experiments = [
    ("TF 50 %", "experience_5_bahdanau"),
    ("TF 100 %", "experience_5_bahdanau_tf100")
]

comparison_rows = []
validation_outputs = {}

for label, folder in experiments:
    checkpoint_path = (
        project_dir / "checkpoints" / folder / "lstm_best.pt"
    )

    saved = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    # Les deux modèles doivent utiliser les mêmes vocabulaires
    assert saved["vocab_fr"] == vocab_fr
    assert saved["vocab_en"] == vocab_en

    # Modèle séparé : le modèle principal reste intact
    eval_encoder = EncoderLSTM(
        len(vocab_fr),
        saved["config"]["embedding_dim"],
        saved["config"]["hidden_dim"],
        saved["config"]["num_layers"]
    )

    eval_decoder = DecoderLSTM(
        len(vocab_en),
        saved["config"]["embedding_dim"],
        saved["config"]["hidden_dim"],
        saved["config"]["num_layers"]
    )

    eval_model = Seq2SeqLSTM(
        eval_encoder, eval_decoder
    ).to(device)

    eval_model.load_state_dict(saved["model_state_dict"])

    rows = []

    for batch in tqdm(validation_loader, desc=label):
        predictions, ended = generate_validation_batch(
            eval_model,
            batch["src"].to(device),
            batch["src_lengths"]
        )

        for source, reference, prediction, has_eos in zip(
            batch["source_texts"],
            batch["reference_texts"],
            predictions,
            ended
        ):
            rows.append({
                "source": source,
                "reference": reference,
                "prediction": prediction,
                "ended_with_eos": has_eos
            })

    df = pd.DataFrame(rows)
    assert len(df) == len(prepared["validation"])

    bleu_metric = BLEU(lowercase=True, tokenize="13a")

    bleu = bleu_metric.corpus_score(
        df["prediction"].tolist(),
        [df["reference"].tolist()]
    )

    comparison_rows.append({
        "configuration": label,
        "best_epoch": saved["epoch"],
        "validation_loss": saved["validation_loss"],
        "validation_bleu": bleu.score,
        "sentences_without_eos": int(
            (~df["ended_with_eos"]).sum()
        ),
        "validation_sentences": len(df),
        "bleu_signature": str(bleu_metric.get_signature())
    })

    validation_outputs[label] = df

    output_dir = project_dir / "results" / folder
    output_dir.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        output_dir / "validation_translations.csv",
        index=False
    )

    del eval_model, eval_encoder, eval_decoder, saved


comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)

print("\nExemples du modèle TF 100 % :")
display(
    validation_outputs["TF 100 %"][
        ["source", "reference", "prediction"]
    ].head(10)
)

TF 50 %:   0%|          | 0/250 [00:00<?, ?it/s]

TF 100 %:   0%|          | 0/250 [00:00<?, ?it/s]

,configuration,best_epoch,validation_loss,validation_bleu,sentences_without_eos,validation_sentences,bleu_signature
0,TF 50 %,4,4.443374,2.950432,125,2000,nrefs:1|case:lc|eff:no|tok:13a|smooth:exp|vers...
1,TF 100 %,3,4.190029,2.441784,235,2000,nrefs:1|case:lc|eff:no|tok:13a|smooth:exp|vers...



Exemples du modèle TF 100 % :


,source,reference,prediction
0,"(en 1991, par exemple, les tribunaux ont réexa...","(for instance, in 1991 the polish courts revie...","( 1991 ) for example , the <UNK> has been <UNK..."
1,ils se sont rencontrés sans moi.,they met without me.,they're <UNK> without me without me .
2,"la ministre villedrouin, a insisté, dans son i...","the minister villedrouin, stressed in his spee...","minister <UNK> , the <UNK> has been in its <UN..."
3,j'ai payé en liquide pour la semaine.,pay cash week .,i have an <UNK> for the week .
4,"en revanche, un amendement du viet nam visant ...","an amendment by viet nam, however, intended to...","in contrast , a third phase of the <UNK> <UNK>..."
5,le certificat peut être imprimé dans une ou pl...,the certifrcate can be printed in one or more ...,the following may be a given or a number of th...
6,59. l'asfc continue à se réserver le droit d'e...,59. the cbsa continues to reserve the right to...,<UNK> . <UNK> <UNK> to the right to the <UNK> ...
7,"le soutien offert par l'apeca, par le biais du...","acoa's support, through the community futures ...","the support for the <UNK> , the program of the..."
8,"l'autre question, d'une portée énorme, est le ...","the other issue, which is of enormous signific...","question , it is a matter of the <UNK> , is ri..."
9,"quoi, la mère porteuse ?","what, the surrogacy?",what are you <UNK> ?


In [16]:
import torch
import pandas as pd
from tqdm.auto import tqdm
from sacrebleu.metrics import BLEU

checkpoint_path = (
    project_dir / "checkpoints"
    / "experience_5_bahdanau" / "lstm_best.pt"
)

saved = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

assert saved["vocab_fr"] == vocab_fr
assert saved["vocab_en"] == vocab_en

cfg = saved["config"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PAD_ID = cfg["special_tokens"]["<PAD>"]
SOS_ID = cfg["special_tokens"]["<SOS>"]
EOS_ID = cfg["special_tokens"]["<EOS>"]

id_to_token_en = {i: token for token, i in vocab_en.items()}

decode_model = Seq2SeqLSTM(
    EncoderLSTM(
        len(vocab_fr), cfg["embedding_dim"],
        cfg["hidden_dim"], cfg["num_layers"]
    ),
    DecoderLSTM(
        len(vocab_en), cfg["embedding_dim"],
        cfg["hidden_dim"], cfg["num_layers"]
    )
).to(device)

decode_model.load_state_dict(saved["model_state_dict"])
decode_model.eval()

print("Modèle TF 50 % chargé — époque", saved["epoch"])


@torch.no_grad()
def translate_no_repeat(src, lengths):
    encoder_outputs, hidden, cell = decode_model.encoder(
        src, lengths
    )

    mask = src.ne(PAD_ID)
    sequences = [[] for _ in range(src.size(0))]
    finished = [False] * src.size(0)

    token = torch.full(
        (src.size(0),), SOS_ID,
        dtype=torch.long, device=src.device
    )

    for _ in range(100):
        logits, hidden, cell, _ = decode_model.decoder(
            token, hidden, cell, encoder_outputs, mask
        )

        for i, sequence in enumerate(sequences):
            if finished[i] or len(sequence) < 2:
                continue

            # Chercher les trigrams déjà produits qui commencent
            # par les deux derniers tokens de la séquence.
            prefix = tuple(sequence[-2:])

            banned = {
                sequence[j + 2]
                for j in range(len(sequence) - 2)
                if tuple(sequence[j:j + 2]) == prefix
            }

            # Toujours autoriser la fin de phrase
            banned.discard(EOS_ID)

            if banned:
                logits[i, list(banned)] = float("-inf")

        next_ids = logits.argmax(dim=-1).tolist()

        for i, index in enumerate(next_ids):
            if finished[i]:
                next_ids[i] = EOS_ID
                continue

            sequences[i].append(index)

            if index == EOS_ID:
                finished[i] = True

        token = torch.tensor(
            next_ids, dtype=torch.long, device=src.device
        )

        if all(finished):
            break

    translations = []

    for sequence in sequences:
        words = []

        for index in sequence:
            if index == EOS_ID:
                break
            if index not in {PAD_ID, SOS_ID}:
                words.append(id_to_token_en.get(index, "<UNK>"))

        translations.append(" ".join(words))

    return translations, finished

Modèle TF 50 % chargé — époque 4


In [17]:
rows = []

for batch in tqdm(validation_loader, desc="Décodage sans trigrammes répétés"):
    predictions, ended = translate_no_repeat(
        batch["src"].to(device),
        batch["src_lengths"]
    )

    for source, reference, prediction, has_eos in zip(
        batch["source_texts"],
        batch["reference_texts"],
        predictions,
        ended
    ):
        rows.append({
            "source": source,
            "reference": reference,
            "prediction": prediction,
            "ended_with_eos": has_eos
        })

no_repeat_df = pd.DataFrame(rows)

assert len(no_repeat_df) == len(prepared["validation"])

metric = BLEU(lowercase=True, tokenize="13a")

score = metric.corpus_score(
    no_repeat_df["prediction"].tolist(),
    [no_repeat_df["reference"].tolist()]
).score

print(f"BLEU validation — TF 50 % initial : {2.9504316936025954:.4f}")
print(f"BLEU validation — sans répétition de trigrammes : {score:.4f}")
print(
    "Phrases sans EOS :",
    int((~no_repeat_df["ended_with_eos"]).sum())
)
print(
    "Longueur moyenne :",
    round(no_repeat_df["prediction"].str.split().map(len).mean(), 2)
)

output_dir = (
    project_dir / "results" / "experience_5_bahdanau"
)
output_dir.mkdir(parents=True, exist_ok=True)

no_repeat_df.to_csv(
    output_dir / "validation_no_repeat_trigram.csv",
    index=False
)

for _, row in no_repeat_df.head(3).iterrows():
    print("\nFR :", row["source"])
    print("Référence :", row["reference"])
    print("Prédiction :", row["prediction"])

Décodage sans trigrammes répétés:   0%|          | 0/250 [00:00<?, ?it/s]

BLEU validation — TF 50 % initial : 2.9504
BLEU validation — sans répétition de trigrammes : 3.8815
Phrases sans EOS : 11
Longueur moyenne : 19.2

FR : (en 1991, par exemple, les tribunaux ont réexaminé 536 cas d'adoption à l'étranger et dans 113 de ces cas (21 %), il leur a été impossible d'établir comment les parents adoptifs avaient choisi l'enfant.
Référence : (for instance, in 1991 the polish courts reviewed 536 foreign adoption cases and in 113 of them (21 per cent) failed to establish how the adoptive parents had chosen the child.
Prédiction : ( 1991 , <UNK> example , example , the <UNK> <UNK> in the cases of 21 ( 21 % ( ( 21 ( ( % ) , the , the the <UNK> of the .

FR : ils se sont rencontrés sans moi.
Référence : they met without me.
Prédiction : they are without without me .

FR : la ministre villedrouin, a insisté, dans son intervention, sur l'image d'haïti à l'extérieur et sur la politique de positionnement d'haïti comme destination touristique.
Référence : the minister vill

In [19]:
from sacremoses import MosesDetokenizer
from sacrebleu.metrics import BLEU

# decode_model est le modèle TF 50 % chargé dans la cellule 21
decode_model.eval()
detokenizer = MosesDetokenizer(lang="en")

final_dir = project_dir / "results" / "experience_5_finale"
export_dir = project_dir / "export_application"

final_dir.mkdir(parents=True, exist_ok=True)
export_dir.mkdir(parents=True, exist_ok=True)

rows = []

for batch in tqdm(test_loader, desc="Évaluation finale du test"):
    predictions, ended = translate_no_repeat(
        batch["src"].to(device),
        batch["src_lengths"]
    )

    for source, reference, prediction, has_eos in zip(
        batch["source_texts"],
        batch["reference_texts"],
        predictions,
        ended
    ):
        rows.append({
            "source": source,
            "reference": reference,
            "prediction_brute": prediction,
            "prediction": detokenizer.detokenize(prediction.split()),
            "ended_with_eos": has_eos
        })

final_df = pd.DataFrame(rows)
assert len(final_df) == len(prepared["test"])

metric = BLEU(lowercase=True, tokenize="13a")
score = metric.corpus_score(
    final_df["prediction"].tolist(),
    [final_df["reference"].tolist()]
)

metrics = {
    "model": "LSTM avec attention de Bahdanau",
    "teacher_forcing_ratio": float(cfg["teacher_forcing_ratio"]),
    "best_epoch": int(saved["epoch"]),
    "decoding": "greedy_no_repeat_trigram",
    "max_generation_steps": 100,
    "detokenization": "MosesDetokenizer(lang=en)",
    "test_bleu": float(score.score),
    "bleu_signature": str(metric.get_signature()),
    "test_sentences": len(final_df),
    "sentences_without_eos": int(
        (~final_df["ended_with_eos"]).sum()
    )
}

final_df.to_csv(
    final_dir / "traductions_test_finales.csv",
    index=False,
    encoding="utf-8"
)

(final_dir / "metriques_finales.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

torch.save(
    {
        "model_state_dict": {
            name: tensor.detach().cpu()
            for name, tensor in decode_model.state_dict().items()
        },
        "config": cfg,
        "vocab_fr": vocab_fr,
        "vocab_en": vocab_en,
        "best_epoch": int(saved["epoch"]),
        "decoding": {
            "method": "greedy",
            "no_repeat_ngram_size": 3,
            "max_generation_steps": 100,
            "detokenization": "moses_en"
        }
    },
    export_dir / "lstm_attention_final2.pt"
)

print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("\nModèle exporté :", export_dir / "lstm_attention_final.pt")

Évaluation finale du test:   0%|          | 0/250 [00:00<?, ?it/s]

{
  "model": "LSTM avec attention de Bahdanau",
  "teacher_forcing_ratio": 0.5,
  "best_epoch": 4,
  "decoding": "greedy_no_repeat_trigram",
  "max_generation_steps": 100,
  "detokenization": "MosesDetokenizer(lang=en)",
  "test_bleu": 3.930902512373017,
  "bleu_signature": "nrefs:1|case:lc|eff:no|tok:13a|smooth:exp|version:2.6.0",
  "test_sentences": 2000,
  "sentences_without_eos": 10
}

Modèle exporté : /content/drive/MyDrive/Colab Notebooks/nlp_translation/export_application/lstm_attention_final.pt
